# Monte Carlo nanodots — Benchmark con observables físicos

Compara la **consistencia física** de tres modos de ejecución midiendo
**E, M, C_v, χ** en función de T para 2 experimentos (helicoidal / skyrmiónico),
e incluye **visualización de espines** en la capa intermedia (colormap jet).

| Modo | Motor | Dispositivo |
|------|-------|-------------|
| 1 | Site-by-site secuencial (NumPy) | CPU |
| 2 | Checkerboard + tensores + `jit` | 1 GPU |
| 3 | Checkerboard + tensores + `pmap` | 2 GPU T4 (Kaggle) |

**Flujo:** corre todo con `MODE=1`, luego cambia a `2`, luego a `3`.
Las últimas celdas superponen resultados y comparan configuraciones visualmente.

> **Ajustado para Kaggle:** Rd = 15.0 → CPU ≤ 11 h; modo 3 preparado para 2× T4.


## 0 · Instalación (Kaggle)

In [ ]:
import os, subprocess, sys

# JAX con CUDA 12 para las T4 de Kaggle
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "--upgrade", "jax[cuda12]", "numpy", "matplotlib"])

# Pre-asignar el 90 % de VRAM por GPU para evitar fragmentación
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"
# Hacer visibles ambas T4 de Kaggle
os.environ["CUDA_VISIBLE_DEVICES"]           = "0,1"


## 1 · Configuración

In [ ]:
# ═══════════════════════════════════════════════
#  CAMBIA SOLO ESTA VARIABLE ENTRE CORRIDAS
# ═══════════════════════════════════════════════
MODE = 2        # 1 = CPU  |  2 = 1 GPU  |  3 = 2 GPUs T4

# ── parámetros de la simulación (iguales en todos los modos) ──
REPLICAS  = 128     # réplicas por experimento
TSTEPS    = 20      # temperaturas en el annealing (Ti → Tf)
MCS_THERM = 50      # barridos de termalización por T
MCS_MEAS  = 50      # barridos de medición por T

# ── geometría ──
Rd = 15.0           # radio del disco (≤ 11 h en CPU; antes 20.3 → 18 h)
L  = 5              # capas en z

# ── constantes ──
So = 1.0
kB = 0.086173404
Ti, Tf = 20.0, 0.1
N_EXPERIMENTS = 2

EXP_PARAMS = [
    dict(kDM=0.20, Jex=1.0, Kan1=0.05, Hex=0.0, gamma=90.0),   # helicoidal
    dict(kDM=1.00, Jex=1.0, Kan1=0.25, Hex=0.0, gamma=90.0),   # skyrmiónico
]

total_batch = N_EXPERIMENTS * REPLICAS
Z_MID = L // 2      # capa intermedia para visualización

print(f"MODE = {MODE}")
print(f"Batch total = {N_EXPERIMENTS} exp × {REPLICAS} réplicas = {total_batch}")
print(f"Por T: {MCS_THERM} therm + {MCS_MEAS} meas sweeps")
print(f"Capa de visualización: z = {Z_MID}  (de 0..{L-1})")


## 2 · Verificar dispositivos

In [ ]:
import jax
print("JAX:", jax.__version__)
print("Backend:", jax.default_backend())
devs = jax.devices()
print(f"Dispositivos: {len(devs)}")
for d in devs:
    print(f"  {d}")
if MODE == 3 and len(devs) < 2:
    print("\n⚠  MODE=3 pero solo hay 1 GPU. pmap correrá en 1 dispositivo.")
if MODE == 1:
    print("\nModo CPU seleccionado — JAX solo se usa para verificación.")


## 3 · Geometría y temperaturas

In [ ]:
import math, time, numpy as np

def build_geometry(Rd, L):
    L2 = int(Rd) + 1
    N  = 2 * L2 + 1
    coord = np.arange(-L2, L2 + 1)
    xc = coord.reshape(1, 1, N)
    yc = coord.reshape(1, N, 1)
    zc = np.arange(1, L + 1).reshape(L, 1, 1)
    disc2d = (xc**2 + yc**2) <= Rd**2
    disco  = np.broadcast_to(disc2d, (L, N, N)).copy()
    checker = (xc + yc + zc) % 2
    even = (checker == 0) & disco
    odd  = (checker == 1) & disco
    return L2, N, disco, even, odd, int(disco.sum())

L2, N, disco_np, even_np, odd_np, N_spins = build_geometry(Rd, L)
temps    = np.linspace(Ti, Tf, TSTEPS)
betas_np = 1.0 / (kB * temps)

print(f"Lattice: {L} × {N} × {N}")
print(f"N_spins = {N_spins}")
print(f"Temperaturas: {Ti} → {Tf} K  ({TSTEPS} pasos)")

est_rate = 70_000
total_up = total_batch * TSTEPS * (MCS_THERM + MCS_MEAS) * N_spins
est_cpu_h = total_up / est_rate / 3600
print(f"\nUpdates totales: {total_up:,.0f}")
print(f"Tiempo estimado CPU: ~{est_cpu_h:.1f} h  (a {est_rate:,} upd/s)")


## 4 · Funciones de observables

Energía total y magnetización, idénticas para CPU y GPU (solo cambia `np` ↔ `jnp`).
Cada enlace se cuenta **una sola vez** (solo vecinos +x, +y, +z).

In [ ]:
def compute_energy_np(spins, disco_mask, kDM, Jex, Kan1, Hex, g):
    """Energía total por configuración. spins (batch, L, Ny, Nx, 3) → (batch,)"""
    S_xp = np.roll(spins, -1, axis=3)
    S_yp = np.roll(spins, -1, axis=2)
    S_zp = np.roll(spins, -1, axis=1)
    S_zp[:, -1] = 0.0

    E_exc = -Jex * np.sum(spins * (S_xp + S_yp + S_zp), axis=-1)
    sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]
    E_an  = (Kan1 / So**4) * ((sx*sy)**2 + (sx*sz)**2 + (sy*sz)**2)
    E_dm  = -kDM * (sy*S_xp[...,2] - sz*S_xp[...,1]
                   + sz*S_yp[...,0] - sx*S_yp[...,2])
    E_z   = -Hex * (sx * np.cos(g) + sz * np.sin(g))
    E_site = (E_exc + E_an + E_dm + E_z) * disco_mask
    return np.sum(E_site, axis=(-3, -2, -1))

def compute_mag_np(spins, disco_mask_3):
    """|M|/N por configuración. spins (batch, L, Ny, Nx, 3) → (batch,)"""
    M_vec = np.sum(spins * disco_mask_3, axis=(1, 2, 3))
    return np.linalg.norm(M_vec, axis=-1) / N_spins

disco_mask_np   = disco_np[None, :, :, :]
disco_mask_3_np = disco_np[None, :, :, :, None]
print("Funciones de observables definidas.")


## 5 · Motor CPU — site-by-site (Modo 1)

Recorre cada sitio secuencialmente. El batch se vectoriza con NumPy.
Retorna observables **y** la configuración final de espines.

In [ ]:
def run_mode_cpu():
    print("═══ MODO 1: CPU clásico (site-by-site, NumPy) ═══")
    sites = list(zip(*np.where(disco_np)))
    rng   = np.random.default_rng(42)

    spins = rng.normal(size=(total_batch, L, N, N, 3))
    spins = spins / np.linalg.norm(spins, axis=-1, keepdims=True) * So
    spins = spins * disco_mask_3_np

    p_kDM = np.array([EXP_PARAMS[i % N_EXPERIMENTS]["kDM"]  for i in range(total_batch)])
    p_Jex = np.array([EXP_PARAMS[i % N_EXPERIMENTS]["Jex"]  for i in range(total_batch)])
    p_Kan = np.array([EXP_PARAMS[i % N_EXPERIMENTS]["Kan1"] for i in range(total_batch)])
    p_Hex = np.array([EXP_PARAMS[i % N_EXPERIMENTS]["Hex"]  for i in range(total_batch)])
    p_g   = np.array([math.radians(EXP_PARAMS[i % N_EXPERIMENTS]["gamma"])
                      for i in range(total_batch)])

    def get_neighbors(sp, z, y, x):
        Lz, Ny, Nx = L, N, N
        S_xp = sp[:, z, y, (x+1)%Nx]; S_xm = sp[:, z, y, (x-1)%Nx]
        S_yp = sp[:, z, (y+1)%Ny, x]; S_ym = sp[:, z, (y-1)%Ny, x]
        S_zp = sp[:, z+1, y, x] if z+1 < Lz else np.zeros((total_batch, 3))
        S_zm = sp[:, z-1, y, x] if z-1 >= 0  else np.zeros((total_batch, 3))
        return S_xp, S_xm, S_yp, S_ym, S_zp, S_zm

    def delta_E(s_old, s_new, nb):
        S_xp, S_xm, S_yp, S_ym, S_zp, S_zm = nb
        dS  = s_new - s_old
        sv  = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
        dE  = -p_Jex * np.sum(dS * sv, axis=-1)
        def an(s):
            return (p_Kan/So**4)*((s[...,0]*s[...,1])**2
                                 +(s[...,0]*s[...,2])**2
                                 +(s[...,1]*s[...,2])**2)
        dE += an(s_new) - an(s_old)
        dE += -p_kDM * (dS[...,1]*(S_xp[...,2]-S_xm[...,2])
                       - dS[...,2]*(S_xp[...,1]-S_xm[...,1])
                       + dS[...,2]*(S_yp[...,0]-S_ym[...,0])
                       - dS[...,0]*(S_yp[...,2]-S_ym[...,2]))
        dE += -p_Hex * (dS[...,0]*np.cos(p_g) + dS[...,2]*np.sin(p_g))
        return dE

    def sweep(sp, beta):
        for z, y, x in sites:
            s_old = sp[:, z, y, x].copy()
            tr = rng.normal(size=(total_batch, 3))
            tr = tr / np.linalg.norm(tr, axis=-1, keepdims=True) * So
            nb = get_neighbors(sp, z, y, x)
            dE = delta_E(s_old, tr, nb)
            acc = (dE < 0) | (rng.random(total_batch) < np.exp(-dE * beta))
            sp[:, z, y, x] = np.where(acc[:, None], tr, s_old)

    obs_all = np.zeros((TSTEPS, 4, total_batch))
    t0 = time.time()

    for ti in range(TSTEPS):
        beta = betas_np[ti]
        elapsed = time.time() - t0
        eta = (elapsed / max(ti, 1)) * (TSTEPS - ti) if ti > 0 else 0
        print(f"  T={temps[ti]:5.1f} K  [{ti+1}/{TSTEPS}]"
              f"  transcurrido {elapsed/3600:.1f}h  ETA {eta/3600:.1f}h", end="\r")

        for _ in range(MCS_THERM):
            sweep(spins, beta)

        sE = np.zeros(total_batch); sE2 = np.zeros(total_batch)
        sM = np.zeros(total_batch); sM2 = np.zeros(total_batch)
        for _ in range(MCS_MEAS):
            sweep(spins, beta)
            for exp in range(N_EXPERIMENTS):
                idx = list(range(exp, total_batch, N_EXPERIMENTS))
                p = EXP_PARAMS[exp]; g_r = math.radians(p["gamma"])
                E = compute_energy_np(spins[idx], disco_mask_np, p["kDM"],
                                      p["Jex"], p["Kan1"], p["Hex"], g_r)
                M = compute_mag_np(spins[idx], disco_mask_3_np)
                sE[idx]  += E;  sE2[idx] += E**2
                sM[idx]  += M;  sM2[idx] += M**2

        obs_all[ti, 0] = sE  / MCS_MEAS
        obs_all[ti, 1] = sE2 / MCS_MEAS
        obs_all[ti, 2] = sM  / MCS_MEAS
        obs_all[ti, 3] = sM2 / MCS_MEAS

    elapsed = time.time() - t0
    print(f"\n  Tiempo total: {elapsed/3600:.2f} h  ({elapsed:.0f} s)")
    return obs_all, elapsed, spins


## 6 · Motor JAX — checkerboard + tensores (Modos 2 y 3)

**Modo 2:** `jax.jit` — todo el batch en 1 GPU.
**Modo 3:** `jax.pmap` — batch repartido entre las 2 T4 de Kaggle.

`run_anneal_obs` retorna **(observables, espines finales)** para poder visualizarlos.

Consideraciones para 2× T4 en Kaggle:
- `CUDA_VISIBLE_DEVICES=0,1` (ya configurado arriba)
- El batch se divide en 2 partes iguales (`total_batch` debe ser par → 256 ✓)
- Cada T4 tiene 15 GB; la carga por GPU es ~128 configs × 5×33×33×3 ≈ 9 MB + intermedios → holgado
- `pmap` sincroniza automáticamente entre GPUs al final de cada `scan` step

In [ ]:
import jax.numpy as jnp
from jax import random, jit, pmap

def build_jax_engine():
    ndev = jax.local_device_count()

    even_j  = jnp.asarray(even_np[None, ..., None])
    odd_j   = jnp.asarray(odd_np[None, ..., None])
    disco_j = jnp.asarray(disco_np[None, ..., None])

    def substep(spins, key, beta, mask, kDM, Jex, Kan1, Hex, g):
        S_xp = jnp.roll(spins, -1, 3); S_xm = jnp.roll(spins, 1, 3)
        S_yp = jnp.roll(spins, -1, 2); S_ym = jnp.roll(spins, 1, 2)
        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)
        S_zm = jnp.roll(spins,  1, 1).at[:,  0].set(0.0)
        kt, ka = random.split(key)
        tr = random.normal(kt, spins.shape)
        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So
        tr = jnp.where(mask, tr, spins)
        dS = tr - spins
        sv = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
        dE = -Jex * jnp.sum(dS * sv, axis=-1)
        def an(s):
            return (Kan1/So**4)*((s[...,0]*s[...,1])**2
                                +(s[...,0]*s[...,2])**2
                                +(s[...,1]*s[...,2])**2)
        dE = dE + an(tr) - an(spins)
        dE = dE - kDM*(dS[...,1]*(S_xp[...,2]-S_xm[...,2])
                      -dS[...,2]*(S_xp[...,1]-S_xm[...,1])
                      +dS[...,2]*(S_yp[...,0]-S_ym[...,0])
                      -dS[...,0]*(S_yp[...,2]-S_ym[...,2]))
        dE = dE - Hex*(dS[...,0]*jnp.cos(g) + dS[...,2]*jnp.sin(g))
        r = random.uniform(ka, dE.shape)
        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask
        return jnp.where(acc, tr, spins)

    def energy_jax(spins, kDM, Jex, Kan1, Hex, g):
        S_xp = jnp.roll(spins, -1, 3)
        S_yp = jnp.roll(spins, -1, 2)
        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)
        E = -Jex * jnp.sum(spins * (S_xp + S_yp + S_zp), axis=-1)
        sx, sy, sz = spins[...,0], spins[...,1], spins[...,2]
        E = E + (Kan1/So**4)*((sx*sy)**2 + (sx*sz)**2 + (sy*sz)**2)
        E = E - kDM*(sy*S_xp[...,2] - sz*S_xp[...,1]
                    + sz*S_yp[...,0] - sx*S_yp[...,2])
        E = E - Hex*(sx*jnp.cos(g) + sz*jnp.sin(g))
        return jnp.sum(E * disco_j[..., 0], axis=(-3,-2,-1))

    def mag_jax(spins):
        M = jnp.sum(spins * disco_j, axis=(1,2,3))
        return jnp.linalg.norm(M, axis=-1) / N_spins

    def run_anneal_obs(spins, key, betas, kDM, Jex, Kan1, Hex, g):
        def per_T(carry, beta):
            s, k = carry
            def therm(_, v):
                s, k = v
                k, k1, k2 = random.split(k, 3)
                s = substep(s, k1, beta, even_j, kDM, Jex, Kan1, Hex, g)
                s = substep(s, k2, beta, odd_j,  kDM, Jex, Kan1, Hex, g)
                return s, k
            s, k = jax.lax.fori_loop(0, MCS_THERM, therm, (s, k))
            b = s.shape[0]
            def meas(_, v):
                s, k, sE, sE2, sM, sM2 = v
                k, k1, k2 = random.split(k, 3)
                s = substep(s, k1, beta, even_j, kDM, Jex, Kan1, Hex, g)
                s = substep(s, k2, beta, odd_j,  kDM, Jex, Kan1, Hex, g)
                E = energy_jax(s, kDM, Jex, Kan1, Hex, g)
                M = mag_jax(s)
                return s, k, sE+E, sE2+E*E, sM+M, sM2+M*M
            s, k, sE, sE2, sM, sM2 = jax.lax.fori_loop(
                0, MCS_MEAS, meas,
                (s, k, jnp.zeros(b), jnp.zeros(b),
                       jnp.zeros(b), jnp.zeros(b)))
            obs = jnp.stack([sE, sE2, sM, sM2]) / MCS_MEAS
            return (s, k), obs
        (s, _), all_obs = jax.lax.scan(per_T, (spins, key), betas)
        return all_obs, s          # ← también retorna espines finales

    return ndev, run_anneal_obs

ndev, run_anneal_obs = build_jax_engine()
print(f"Motor JAX construido  |  GPUs visibles: {ndev}")


## 7 · Ejecución

In [ ]:
def make_params_jax(batch):
    def col(name, transform=lambda v: v):
        vals = [transform(EXP_PARAMS[i % N_EXPERIMENTS][name])
                for i in range(batch)]
        return jnp.asarray(np.array(vals).reshape(batch, 1, 1, 1))
    return col("kDM"), col("Jex"), col("Kan1"), col("Hex"), \
           col("gamma", transform=math.radians)

def init_spins_jax(batch, seed=1):
    k = random.PRNGKey(seed)
    s = random.normal(k, (batch, L, N, N, 3))
    s = s / jnp.linalg.norm(s, axis=-1, keepdims=True) * So
    return s * jnp.asarray(disco_np[None, :, :, :, None])

betas_j = jnp.asarray(betas_np)
key     = random.PRNGKey(42)

if MODE == 1:
    obs_raw, elapsed, final_spins_np = run_mode_cpu()

elif MODE == 2:
    print("═══ MODO 2: 1 GPU (jit) ═══")
    run_fn = jit(run_anneal_obs)
    pK, pJ, pA, pH, pG = make_params_jax(total_batch)
    spins = init_spins_jax(total_batch)

    print("  Compilando JIT ...", end=" ", flush=True)
    out = run_fn(spins, key, betas_j, pK, pJ, pA, pH, pG)
    out[0].block_until_ready()
    print("OK")

    print("  Ejecutando ...", flush=True)
    t0 = time.time()
    obs, final_spins = run_fn(spins, key, betas_j, pK, pJ, pA, pH, pG)
    obs.block_until_ready()
    elapsed = time.time() - t0
    obs_raw = np.asarray(obs)
    final_spins_np = np.asarray(final_spins)
    print(f"  Tiempo: {elapsed:.1f} s")

elif MODE == 3:
    actual_ndev = jax.local_device_count()
    print(f"═══ MODO 3: {actual_ndev} GPUs (pmap) ═══")

    per_dev = math.ceil(total_batch / actual_ndev)
    b = per_dev * actual_ndev
    print(f"  {total_batch} configs → {per_dev} por GPU × {actual_ndev} = {b}")

    run_fn = pmap(run_anneal_obs, in_axes=(0, 0, None, 0, 0, 0, 0, 0))

    pK, pJ, pA, pH, pG = make_params_jax(b)
    r5 = lambda a: a.reshape(actual_ndev, per_dev, 1, 1, 1)
    pK, pJ, pA, pH, pG = r5(pK), r5(pJ), r5(pA), r5(pH), r5(pG)

    spins = init_spins_jax(b).reshape(actual_ndev, per_dev, L, N, N, 3)
    keys  = random.split(key, actual_ndev)

    print("  Compilando pmap ...", end=" ", flush=True)
    out = run_fn(spins, keys, betas_j, pK, pJ, pA, pH, pG)
    out[0].block_until_ready()
    print("OK")

    print("  Ejecutando ...", flush=True)
    t0 = time.time()
    obs, final_s = run_fn(spins, keys, betas_j, pK, pJ, pA, pH, pG)
    obs.block_until_ready()
    elapsed = time.time() - t0

    # obs: (ndev, TSTEPS, 4, per_dev) → (TSTEPS, 4, b)
    obs_raw = np.asarray(obs).transpose(1, 2, 0, 3).reshape(TSTEPS, 4, b)
    obs_raw = obs_raw[:, :, :total_batch]
    # spins: (ndev, per_dev, L, N, N, 3) → (b, L, N, N, 3)
    final_spins_np = np.asarray(final_s).reshape(b, L, N, N, 3)[:total_batch]
    print(f"  Tiempo: {elapsed:.1f} s")

print(f"\n✓ Modo {MODE} completado en {elapsed:.1f} s ({elapsed/3600:.2f} h)")
print(f"  Forma espines finales: {final_spins_np.shape}")


## 8 · Calcular E, M, C_v, χ por experimento

- **C_v** = (⟨E²⟩ − ⟨E⟩²) / (k_B T² · N_spins)
- **χ** = (⟨M²⟩ − ⟨M⟩²) / (k_B T)

In [ ]:
results = {}

for exp in range(N_EXPERIMENTS):
    idx = list(range(exp, total_batch, N_EXPERIMENTS))
    avg_E  = obs_raw[:, 0, idx].mean(axis=1)
    avg_E2 = obs_raw[:, 1, idx].mean(axis=1)
    avg_M  = obs_raw[:, 2, idx].mean(axis=1)
    avg_M2 = obs_raw[:, 3, idx].mean(axis=1)

    var_E = avg_E2 - avg_E**2
    var_M = avg_M2 - avg_M**2

    results[exp] = dict(
        E_per_spin = avg_E / N_spins,
        M_per_spin = avg_M,
        Cv         = var_E / (kB * temps**2) / N_spins,
        chi        = var_M / (kB * temps),
    )

    tag = ["Helicoidal", "Skyrmiónico"][exp]
    r = results[exp]
    print(f"\n{'='*40}")
    print(f"  Exp {exp}: {tag}")
    print(f"{'='*40}")
    print(f"  E/N   @ 20 K : {r['E_per_spin'][0]:.4f}")
    print(f"  E/N   @ {Tf} K: {r['E_per_spin'][-1]:.4f}")
    print(f"  |M|/N @ 20 K : {r['M_per_spin'][0]:.4f}")
    print(f"  |M|/N @ {Tf} K: {r['M_per_spin'][-1]:.4f}")
    print(f"  Cv max        : {r['Cv'].max():.4f}  @ T={temps[r['Cv'].argmax()]:.1f} K")
    print(f"  χ  max        : {r['chi'].max():.6f}  @ T={temps[r['chi'].argmax()]:.1f} K")


## 9 · Guardar resultados (observables + espines capa intermedia)

In [ ]:
import os

# Extraer solo la capa intermedia para visualización (ahorra espacio)
# Forma: (total_batch, N, N, 3)
mid_layer_spins = final_spins_np[:, Z_MID, :, :, :]

fname = f"results_mode{MODE}.npz"
np.savez(fname,
         mode=MODE, temps=temps, N_spins=N_spins, elapsed=elapsed,
         Rd=Rd, L=L, Z_MID=Z_MID, REPLICAS=REPLICAS,
         MCS_THERM=MCS_THERM, MCS_MEAS=MCS_MEAS,
         disco_2d=disco_np[Z_MID],        # máscara 2D del disco
         mid_layer_spins=mid_layer_spins,  # espines capa z_mid
         **{f"exp{e}_{k}": v for e in range(N_EXPERIMENTS)
            for k, v in results[e].items()})
print(f"→ {fname}  ({os.path.getsize(fname) / 1e6:.1f} MB)")
print(f"  Incluye espines de capa z={Z_MID}: {mid_layer_spins.shape}")
print(f"  Corre de nuevo con otro MODE y las gráficas compararán todos.")


## 10 · Gráfica comparativa de observables

Carga todos los `results_mode*.npz` disponibles y superpone E(T), M(T), Cv(T), χ(T).

In [ ]:
import matplotlib.pyplot as plt

all_modes = {}
for m in [1, 2, 3]:
    fn = f"results_mode{m}.npz"
    if os.path.exists(fn):
        all_modes[m] = dict(np.load(fn, allow_pickle=True))
        print(f"  Cargado {fn}  (modo {m}, {float(all_modes[m]['elapsed']):.0f} s)")

mode_labels = {1: "CPU clásico", 2: "1 GPU (jit)", 3: "2 GPUs (pmap)"}
mode_colors = {1: "#c0392b",     2: "#2980b9",     3: "#27ae60"}
mode_styles = {1: "o-",          2: "s--",          3: "^:"}
exp_names   = ["Helicoidal", "Skyrmiónico"]
obs_info    = [("E_per_spin", "E / N"),
               ("M_per_spin", "|M| / N"),
               ("Cv",         "$C_v$ (por espín)"),
               ("chi",        "$\\chi$ (por espín)")]

fig, axes = plt.subplots(N_EXPERIMENTS, 4,
                         figsize=(18, 4.5 * N_EXPERIMENTS), squeeze=False)

for exp in range(N_EXPERIMENTS):
    for j, (okey, olabel) in enumerate(obs_info):
        ax = axes[exp, j]
        for m, data in sorted(all_modes.items()):
            T = data["temps"]
            y = data[f"exp{exp}_{okey}"]
            ax.plot(T, y, mode_styles[m], color=mode_colors[m],
                    label=mode_labels[m], markersize=4, linewidth=1.2)
        ax.set_xlabel("T (K)")
        ax.set_ylabel(olabel)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        if j == 0:
            ax.set_ylabel(f"{exp_names[exp]}\n{olabel}")

fig.suptitle("Consistencia de observables físicos entre modos MC", fontsize=14)
fig.tight_layout()
fig.savefig("comparacion_observables.png", dpi=150, bbox_inches="tight")
plt.show()
print("→ comparacion_observables.png")

if len(all_modes) >= 2:
    print("\n" + "="*50)
    print(f"  {'Modo':<20}{'Tiempo':>12}{'Speedup':>10}")
    print("="*50)
    t_ref = max(float(d["elapsed"]) for d in all_modes.values())
    for m in sorted(all_modes):
        t = float(all_modes[m]["elapsed"])
        su = t_ref / t if t > 0 else 0
        if t < 120:    ts = f"{t:.1f} s"
        elif t < 7200: ts = f"{t/60:.1f} min"
        else:          ts = f"{t/3600:.1f} h"
        print(f"  {mode_labels[m]:<20}{ts:>12}{su:>9.0f}×")


## 11 · Visualización de espines — capa intermedia (jet)

Muestra **S_z** en la capa z intermedia para la **réplica 0** de cada experimento,
comparando lado a lado todos los modos disponibles. El colormap `jet` va de
azul (S_z = −1, espín apuntando −z) a rojo (S_z = +1, espín apuntando +z).

Debajo se añade un mapa de flechas **(S_x, S_y)** sobre el fondo de S_z
para ver la textura de espín completa (fase helicoidal vs skyrmiónica).

In [ ]:
exp_names = ["Helicoidal", "Skyrmiónico"]

n_modes = len(all_modes)
fig, axes = plt.subplots(N_EXPERIMENTS, max(n_modes, 1),
                         figsize=(5.5 * max(n_modes, 1), 5.2 * N_EXPERIMENTS),
                         squeeze=False)

# Máscara del disco para la capa intermedia
disco_2d = disco_np[Z_MID]  # (N, N)
mask_nan = np.where(disco_2d, 1.0, np.nan)

for col_i, m in enumerate(sorted(all_modes)):
    data = all_modes[m]
    spins_mid = data["mid_layer_spins"]  # (total_batch, N, N, 3)

    for exp in range(N_EXPERIMENTS):
        ax = axes[exp, col_i]

        # Réplica 0 de este experimento
        rep_idx = exp  # config 0=exp0, 1=exp1, 2=exp0, 3=exp1, ...
        Sx = spins_mid[rep_idx, :, :, 0] * mask_nan
        Sy = spins_mid[rep_idx, :, :, 1] * mask_nan
        Sz = spins_mid[rep_idx, :, :, 2] * mask_nan

        # Fondo: S_z con colormap jet
        im = ax.imshow(Sz, origin="lower", cmap="jet",
                       vmin=-So, vmax=So, interpolation="nearest")

        # Flechas: (Sx, Sy) sub-muestreadas
        step = max(1, N // 16)
        yy, xx = np.mgrid[0:N:step, 0:N:step]
        ax.quiver(xx, yy,
                  Sx[::step, ::step], Sy[::step, ::step],
                  color="k", alpha=0.6, scale=25, width=0.004,
                  headwidth=3, headlength=4)

        ax.set_title(f"{exp_names[exp]}  —  {mode_labels[m]}", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="$S_z$")

fig.suptitle(f"Espines en capa intermedia (z = {Z_MID})  —  T final = {Tf} K",
             fontsize=14)
fig.tight_layout()
fig.savefig("visualizacion_espines.png", dpi=150, bbox_inches="tight")
plt.show()
print("→ visualizacion_espines.png")
